# Free Photo-to-Talking-Video (Hindi audio) — SadTalker

This notebook attempts open-source SadTalker inference using one portrait image and a pre-generated Hindi audio file. It includes a small compatibility patch for a known BasicSR/TorchVision import issue. The upstream project documents Python 3.8 as its tested baseline; current hosted notebook runtimes can differ, so this is best-effort rather than guaranteed. Free GPU access is not guaranteed.

**Input:** one portrait photo + Hindi narration audio. **Output:** talking-photo MP4 only if inference succeeds and a non-empty MP4 is validated. This does not translate text into Hindi audio.

In [ ]:
import os, subprocess, sys
print('Python:', sys.version)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch check:', repr(e))
from google.colab import files
uploaded = files.upload()  # Select one portrait image and one Hindi narration audio file
print('Uploaded:', list(uploaded.keys()))
from pathlib import Path
images = [p for p in Path('/content').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
audios = [p for p in Path('/content').iterdir() if p.is_file() and p.suffix.lower() in {'.wav','.mp3','.m4a','.ogg'}]
print('Image candidates:', [p.name for p in images])
print('Audio candidates:', [p.name for p in audios])
assert images, 'Upload a portrait image (.jpg/.png/.webp).'
assert audios, 'Upload narration audio (.wav/.mp3/.m4a/.ogg).'


## Install SadTalker
Run the next cell once. This downloads the source and model checkpoints; it may take several minutes.

In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil
repo = Path('/content/SadTalker')
def run_checked(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)
if repo.exists() and (repo / 'inference.py').is_file():
    print('SadTalker checkout already exists; reusing it.')
else:
    if repo.exists(): shutil.rmtree(repo)
    run_checked(['git','clone','--depth','1','https://github.com/OpenTalker/SadTalker.git',repo])
print('Installing ffmpeg system package...')
run_checked(['bash','-lc','apt-get update -qq && apt-get install -y -qq ffmpeg'])
print('Installing compatible common numeric/audio dependencies first...')
run_checked([sys.executable,'-m','pip','install','-q','numpy<2','librosa','soundfile'])
print('Installing SadTalker requirements. If this fails, stop and copy the full red error.')
run_checked([sys.executable,'-m','pip','install','-q','-r',repo/'requirements.txt'])
print('Applying a narrow BasicSR/TorchVision compatibility patch if the known old import exists...')
import site
patch_candidates = []
for site_dir in site.getsitepackages():
    patch_candidates.extend(Path(site_dir).glob('basicsr/data/degradations.py'))
patch_applied = False
for patch_file in patch_candidates:
    patch_text = patch_file.read_text()
    old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
    new = 'from torchvision.transforms.functional import rgb_to_grayscale'
    if old in patch_text:
        patch_file.write_text(patch_text.replace(old, new))
        print('Applied BasicSR import compatibility patch:', patch_file)
        patch_applied = True
if not patch_applied:
    print('No matching installed BasicSR file/import found; continuing without patch.')
checkpoint_dir = repo/'checkpoints'
print('Downloading SadTalker model checkpoints...')
run_checked(['bash','scripts/download_models.sh'], cwd=repo)
if not checkpoint_dir.exists() or not any(checkpoint_dir.rglob('*')):
    raise RuntimeError('Model checkpoint directory is empty after download. See download logs above.')
print('Setup cell completed. Verify input files, then run inference.')


In [ ]:
from pathlib import Path
import subprocess, sys, shutil
files_now = [p for p in Path('/content').iterdir() if p.is_file()]
print('Files in /content:', [p.name for p in files_now])
image_path = input('Enter exact uploaded portrait filename (e.g. portrait.jpg): ').strip()
audio_path = input('Enter exact uploaded Hindi audio filename (e.g. hindi.wav): ').strip()
image_file = Path('/content', image_path)
audio_file = Path('/content', audio_path)
assert image_file.is_file() and image_file.stat().st_size > 0, f'Photo missing/empty: {image_file}'
assert audio_file.is_file() and audio_file.stat().st_size > 0, f'Audio missing/empty: {audio_file}'
out_dir = '/content/SadTalker/results/kiran_talking_photo'
Path(out_dir).mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'inference.py', '--driven_audio', str(audio_file), '--source_image', str(image_file), '--result_dir', out_dir, '--still', '--preprocess', 'crop']
print('Running SadTalker; this can take several minutes...')
try:
    subprocess.run(cmd, cwd='/content/SadTalker', check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f'SadTalker inference exited with code {e.returncode}. Read the first error above; do not treat this run as successful.') from e
videos = sorted(Path(out_dir).rglob('*.mp4'), key=lambda p: p.stat().st_mtime, reverse=True)
videos = [p for p in videos if p.stat().st_size > 1000]
assert videos, 'No MP4 produced. Check inference traceback, GPU/runtime compatibility, and model downloads.'
output_path = Path('/content/talking_photo_hindi.mp4')
shutil.copy2(videos[0], output_path)
assert output_path.stat().st_size > 1000, 'Output MP4 is empty or incomplete.'
print('VALIDATED OUTPUT:', output_path, 'bytes:', output_path.stat().st_size)
from IPython.display import Video, display
display(Video(str(output_path), embed=True))


In [ ]:
from google.colab import files
assert Path('/content/talking_photo_hindi.mp4').is_file() and Path('/content/talking_photo_hindi.mp4').stat().st_size > 1000
files.download('/content/talking_photo_hindi.mp4')

## Troubleshooting
- `CUDA available: False` means this session has no GPU; in Colab choose Runtime → Change runtime type → T4 GPU if offered, then rerun from the top. Free GPU access is not guaranteed.
- If cloning fails, check the runtime internet connection and rerun the setup cell.
- If `pip install` fails, capture the first dependency error; current Colab Python versions may conflict with older upstream dependencies. Do not ignore the error and continue.
- If model download fails, rerun the setup cell and check the first download error. The checkpoint directory must exist before inference.
- If inference fails, copy the complete final traceback. If no non-empty MP4 is found, this run has failed; no fake still-image output is substituted.
- SadTalker is open-source, but free GPU availability, runtime duration, and hosting are not guaranteed.